In [1]:
import pandas as pd
import subprocess
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
import sys
from sklearn.metrics import precision_recall_curve, auc
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# Analysis Configuration

In [2]:
figure_date_of_record='20250507'
only_2018 = True #Use only variants last evaluated in 2018 at the earliest?
intron_subset = False #Subset for deep and near intronic variants?
save_plots=False #Save plots?

# ClinVar summary pre-processing

In [3]:
clinvar_file='/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/variant_summary_2025-01.txt' #ClinVar Jan 2025 variant summary

In [4]:
original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file

/tmp/16795745.1.shendure-login.q/ipykernel_874957/1145797189.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file


In [5]:
clinvar_df = original_clinvar_df[['Type','Name', 'GeneSymbol', 'Assembly', 'Chromosome', 'Start', 'Stop', 'ClinicalSignificance', 'ReviewStatus', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'LastEvaluated']] #Grab desired columns
clinvar_df = clinvar_df.loc[(clinvar_df['Assembly']=='GRCh38') & (clinvar_df['Type']== 'single nucleotide variant') & (clinvar_df['Name'].str.contains(':'))].copy() #Filter for hg19 and SNVs only

In [6]:
filtered_full_clinvar_df = clinvar_df[clinvar_df['ReviewStatus'].isin(['criteria provided', 'multiple submitters, no conflicts', 'criteria provided, conflicting classifications', 'criteria provided, single submitter', 'reviewed by expert panel', 'practice guideline'])].copy() #Filter for 1-star plus
filtered_full_clinvar_df['YearLastEvaluated']=filtered_full_clinvar_df['LastEvaluated'].transform(lambda x: str(x)[-4:])
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,LastEvaluated,YearLastEvaluated
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A,"Jul 05, 2022",2022
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T,"Nov 01, 2024",2024
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C,"Aug 25, 2021",2021
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C,"Jul 24, 2024",2024
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A,"Sep 11, 2018",2018


In [7]:
filtered_full_clinvar_df['hgvs_c']=filtered_full_clinvar_df['Name'].transform(lambda x: x.split(':')[1]) #Pull hgvs_c from Name column
filtered_full_clinvar_df['aa_change']=filtered_full_clinvar_df['hgvs_c'].transform(lambda x: x.split(' ')[1][1:-1] if 'p.' in x else np.nan) #Gets amino acid change if applicable
filtered_full_clinvar_df['pos_id']=filtered_full_clinvar_df['GeneSymbol'] + ':' + filtered_full_clinvar_df['Start'].astype(str) + ':' + filtered_full_clinvar_df['AlternateAlleleVCF'] #Make variant ID field
filtered_full_clinvar_df['broad_consequence'] = filtered_full_clinvar_df['aa_change'].transform( 
    lambda x: 'non_coding' if (pd.isna(x) or 'p.' not in x)
              else 'synonymous' if '=' in x
              else 'non_synonymous'
) #broadly classsifies molecular consequence

filtered_full_clinvar_df['intronic_dist'] = filtered_full_clinvar_df['hgvs_c'].str.extract(r'\d[+-](\d+)') #Extracts distance from coding sequence
filtered_full_clinvar_df['intronic_dist']=filtered_full_clinvar_df['intronic_dist'].fillna(0)
filtered_full_clinvar_df['intronic_dist']=filtered_full_clinvar_df['intronic_dist'].astype(int)
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,LastEvaluated,YearLastEvaluated,hgvs_c,aa_change,pos_id,broad_consequence,intronic_dist
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A,"Jul 05, 2022",2022,c.166G>A (p.Gly56Arg),p.Gly56Arg,NUBPL:31562125:A,non_synonymous,0
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T,"Nov 01, 2024",2024,c.193A>T (p.Ser65Cys),p.Ser65Cys,HFE:26090957:T,non_synonymous,0
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C,"Aug 25, 2021",2021,c.314T>C (p.Ile105Thr),p.Ile105Thr,HFE:26091078:C,non_synonymous,0
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C,"Jul 24, 2024",2024,c.277G>C (p.Gly93Arg),p.Gly93Arg,HFE:26091041:C,non_synonymous,0
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A,"Sep 11, 2018",2018,c.892+48G>A,NaN,HFE:26093008:A,non_coding,48


In [8]:
syn_noncoding_clinvar_df = filtered_full_clinvar_df[filtered_full_clinvar_df['broad_consequence'].isin(['synonymous', 'non_coding'])].copy() #Gets generally synonymous and non-coding variants (includes intronic) mostly
syn_noncoding_clinvar_df = syn_noncoding_clinvar_df[syn_noncoding_clinvar_df['ClinicalSignificance'].isin(['Benign', 'Pathogenic','Likely pathogenic', 'Pathogenic; drug response', 'Likely pathogenic; drug response', 
                                                                                                           'Likely benign','Benign/Likely benign', 'risk factor','Pathogenic/Likely pathogenic'])].copy() #Filters out VUS
if only_2018:
    syn_noncoding_clinvar_df = syn_noncoding_clinvar_df[syn_noncoding_clinvar_df['YearLastEvaluated'] != '-']
    syn_noncoding_clinvar_df['YearLastEvaluated'] = syn_noncoding_clinvar_df['YearLastEvaluated'].astype(int)
    syn_noncoding_clinvar_df = syn_noncoding_clinvar_df[syn_noncoding_clinvar_df['YearLastEvaluated'] <= 2018]
    syn_noncoding_clinvar_df.head()

In [9]:
#Renames ClinVar consequences
syn_noncoding_clinvar_df.loc[syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('enign'), 'ClinicalSignificance'] = 'BLB'
syn_noncoding_clinvar_df.loc[(syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('hogenic')) | (syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('risk factor')), 'ClinicalSignificance'] = 'PLP'

for_vcf = syn_noncoding_clinvar_df.rename(columns = {'Chromosome': 'chrom', 
                                                     'PositionVCF': 'pos',
                                                     'ReferenceAlleleVCF': 'ref',
                                                     'AlternateAlleleVCF': 'alt'
                                                    }
                                         ) #Renames clinvar for VCF building

for_vcf = for_vcf[~for_vcf['chrom'].isin(['MT', 'Un'])] #Drops other chromosomes

#Some error checking and filtering for weird rows
for_vcf=for_vcf[for_vcf['pos']>0]
for_vcf=for_vcf[for_vcf['alt']!=for_vcf['ref']]
for_vcf = for_vcf.sort_values(["chrom", "pos"])

plp_vcf_df = for_vcf[for_vcf['ClinicalSignificance']=='PLP']

In [10]:
#Saves VCF if needed
save_vcf=False
if save_vcf:
    with open('/net/bbi/vol1/home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vcf.vcf', 'w') as f:
        f.write("##fileformat=VCFv4.3\n")
        f.write("##reference=GRCh38\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in for_vcf.iterrows():
            f.write(
                f"{row['chrom']}\t{row['pos']}\t.\t"
                f"{row['ref'].upper()}\t{row['alt'].upper()}\t.\tPASS\t.\n"
            )

# Analysis Start

In [11]:
vcf_path = '/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vep_combined.txt'
raw_vep_df = pd.read_csv(vcf_path, sep='\t')

raw_vep_df['pos']=raw_vep_df['Location'].transform(lambda x: int(x.split(':')[1].split('-')[0]))
raw_vep_df['pos_id']=raw_vep_df['SYMBOL'] + ':' + raw_vep_df['pos'].astype(str) + ':' +  raw_vep_df['Allele']
raw_vep_df['first_consequence']=raw_vep_df['Consequence'].transform(lambda x: x.split(',')[0])

raw_vep_df['maxSpliceAI'] = raw_vep_df[['SpliceAI_pred_DS_AG', 'SpliceAI_pred_DS_AL', 'SpliceAI_pred_DS_DG', 'SpliceAI_pred_DS_DL']].max(axis=1)

In [12]:
vep_df = raw_vep_df[~raw_vep_df['maxSpliceAI'].isin(['-'])].copy()
vep_df['maxSpliceAI'] = vep_df['maxSpliceAI'].astype(float)
vep_df['splice_impact']=vep_df[['SpliceAI_pred_DS_AG', 'SpliceAI_pred_DS_AL', 'SpliceAI_pred_DS_DG', 'SpliceAI_pred_DS_DL']].idxmax(axis=1)
vep_df.loc[vep_df['maxSpliceAI'] < 0.2, 'splice_impact'] = '< Threshold'

vep_df = vep_df.replace({'SpliceAI_pred_DS_AG': 'Acceptor Gain',
        'SpliceAI_pred_DS_AL': 'Acceptor Loss',
        'SpliceAI_pred_DS_DG': 'Donor Gain',
        'SpliceAI_pred_DS_DL': 'Donor Loss'})

vep_df['simple_splice_impact'] = '< Threshold'
vep_df.loc[vep_df['maxSpliceAI'].astype(float) >= 0.2, 'simple_splice_impact'] = 'Splice Impact Predicted'

vep_df = vep_df[['SYMBOL', 'Consequence', 'first_consequence', 'pos', 'pos_id', 'maxSpliceAI', 'splice_impact', 'simple_splice_impact']]

vep_df = vep_df.rename(columns={'SYMBOL':'GeneSymbol'})
vep_df.head()

,GeneSymbol,Consequence,first_consequence,pos,pos_id,maxSpliceAI,splice_impact,simple_splice_impact
1,SAMD11,synonymous_variant,synonymous_variant,925956,SAMD11:925956:T,0.03,< Threshold,< Threshold
2,SAMD11,synonymous_variant,synonymous_variant,925980,SAMD11:925980:T,0.11,< Threshold,< Threshold
3,SAMD11,synonymous_variant,synonymous_variant,925986,SAMD11:925986:T,0.01,< Threshold,< Threshold
4,SAMD11,synonymous_variant,synonymous_variant,926010,SAMD11:926010:T,0.00,< Threshold,< Threshold
5,SAMD11,intron_variant,intron_variant,926025,SAMD11:926025:A,0.00,< Threshold,< Threshold


In [13]:
vep_df_filtered = vep_df[(vep_df['Consequence'].str.contains('intron_variant')) | (vep_df['Consequence'].str.contains('synonymous'))] #Filters for any variant whose annotation contains intron or synonymous. This may include variants with splice region
vep_df_filtered_splice = vep_df[(vep_df['Consequence'].str.contains('splice')) & (vep_df['Consequence'].str.contains('intron'))] #Filters for only variants in the splice region
vep_df_filtered_canonical_splice = vep_df[(vep_df['Consequence'].str.contains('splice_acceptor_variant')) | (vep_df['Consequence'].str.contains('splice_donor_variant'))] #Filters for only variants in the splice region
vep_df_strict_filtered=vep_df[vep_df['first_consequence'].isin(['intron_variant', 'synonymous_variant'])] #Filters for pure intronic and synonymous variants only

dfs_for_analysis = {'all_non_coding_excl_splicesite': vep_df_filtered,
                    'splice_region_only': vep_df_filtered_splice,
                    'intron_synonymous only': vep_df_strict_filtered
                   }

## Analysis Helper Functions

In [14]:
def consequence_bars(df, filter_type):

    grouped = df.groupby(['first_consequence', 'ClinicalSignificance'])

    proportion_df = []
    
    for group, subset in grouped:
        group_df = pd.concat([subset.value_counts(subset=['first_consequence', 'ClinicalSignificance', 'simple_splice_impact']), subset.value_counts(subset=['first_consequence', 'ClinicalSignificance', 'simple_splice_impact'],normalize=True)], axis=1, keys=['count', 'proportion']).reset_index()
        group_df['total_count'] = len(subset)
        proportion_df.append(group_df)

    proportion_df = pd.concat(proportion_df)

    bar = alt.Chart(proportion_df).mark_bar().encode(
        x=alt.X('first_consequence:N'),
        y=alt.Y('proportion:Q'),
        color=alt.Color('simple_splice_impact:N')
    ).properties(width =300, height = 400).facet('ClinicalSignificance')
    
    bar.display()

    return bar, proportion_df

In [15]:
def make_pr_curve(
    df: pd.DataFrame,
    score_col: str = "maxSpliceAI",
    label_col: str = "ClinicalSignificance",
    pos_label: str = "PLP",
    neg_label: str = "BLB",
    title: str = "Precision-Recall",
) -> alt.Chart:
    """Build a Precision-Recall curve chart using Altair.

    Rows with labels other than pos_label/neg_label (e.g. 'VUS')
    are excluded.

    Parameters
    ----------
    df : DataFrame containing score_col and label_col.
    score_col : Column with continuous predictor scores.
    label_col : Column with functional class labels.
    pos_label : Label string treated as the positive class.
    neg_label : Label string treated as the negative class.
    title : Chart title prefix; AUC-PR is appended automatically.

    Returns
    -------
    alt.Chart
    """

    clinvar_vars = len(df.dropna(subset=[label_col]))

    sub = df[df[label_col].isin([pos_label, neg_label])].copy()

    if len(sub) < 10:
        return None

    y_true = (sub[label_col] == pos_label).astype(int)

    y_score = sub[score_col]

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    baseline = y_true.mean()

    curve_df = pd.DataFrame({"Recall": recall, "Precision": precision})

    curve = (
        alt.Chart(curve_df, title=f"{title} (n = {clinvar_vars})")
        .mark_line(color="orange")
        .encode(
            x=alt.X("Recall:Q", scale=alt.Scale(domain=[0, 1]),
                    axis=alt.Axis(title="Recall", labelFontSize = 18, titleFontSize = 20, labelFont="Arial", titleFont="Arial")),
            y=alt.Y("Precision:Q", scale=alt.Scale(domain=[baseline * 0.95, 1]),
                    axis=alt.Axis(title="Precision", labelFontSize = 18, titleFontSize = 20, labelFont="Arial", titleFont="Arial")),
            tooltip=[
                alt.Tooltip("Recall:Q", format=".3f"),
                alt.Tooltip("Precision:Q", format=".3f"),
            ],
        )
        .properties(width=300, height=300)
    )

    baseline_df = pd.DataFrame({"y": [baseline]})
    baseline_rule = (
        alt.Chart(baseline_df)
        .mark_rule(color="gray", strokeDash=[4, 4])
        .encode(y="y:Q")
    )

    auc_text = alt.Chart(pd.DataFrame({
        'x': [0.05],
        'y': [baseline * 1.1],
        'text': [f'AUC = {pr_auc:.3f}']
    })).mark_text(
        align='left',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x=alt.X('x:Q', scale=alt.Scale(domain=[0, 1])),
        y=alt.Y('y:Q', scale=alt.Scale(domain=[baseline * 0.95, 1])),
        text='text:N'
    )

    final_plot = (curve + baseline_rule + auc_text).configure_title(font="Arial", fontSize=14).configure_axis(grid = False).configure_view(stroke = None)
    final_plot.display()
    return final_plot


In [16]:
def heatmap(df):


    plp_df = df.loc[(df['ClinicalSignificance'] == 'PLP') & (df['simple_splice_impact'] == 'Splice Impact Predicted')]
    blb_df = df.loc[(df['ClinicalSignificance'] == 'BLB') & (df['simple_splice_impact'] == 'Splice Impact Predicted')]

    formap = pd.concat([plp_df, blb_df])
    print(formap)
    formap['pct'] = formap['proportion'] * 100

    base = alt.Chart(formap).encode(
        x=alt.X('ClinicalSignificance:N'),
        y=alt.Y('first_consequence:N')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('pct:Q',
                       scale=alt.Scale(scheme='oranges', domain=[0, 100]),
                       legend=alt.Legend(title='% predicted')
                    )
    )

    pct_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('pct:Q', format='.1f'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )

    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"n=" + datum.total_count'
    ).encode(
        text=alt.Text('n_label:N'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )
    
    proportion_map = (heatmap + pct_text + n_text).properties(width = 150, height = 250)
    proportion_map.display()

    return proportion_map

## Batched Analysis

In [17]:
final_df = pd.merge(for_vcf, vep_df, on=['pos_id', 'pos', 'GeneSymbol'], how='inner')[['Name', 'GeneSymbol', 'chrom', 'ClinicalSignificance', 'ReviewStatus', 'pos', 'ref', 'alt', 'hgvs_c', 'aa_change', 'pos_id', 'Consequence','first_consequence','intronic_dist', 'maxSpliceAI', 'splice_impact', 'simple_splice_impact', 'YearLastEvaluated']]

to_filter = {'all_excl_splite_site': ['synonymous_variant', 'intron_variant', 'splice_polypyrimidine_tract_variant', 'splice_region_variant', 'splice_donor_region_variant', 'splice_donor_5th_base_variant'],
             'splice_region_only': ['splice_polypyrimidine_tract_variant', 'splice_region_variant', 'splice_donor_region_variant', 'splice_donor_5th_base_variant'],
             'intron_synonymous_only': ['synonymous_variant', 'intron_variant']
            }

In [18]:
for key in to_filter.keys():

    filtered_df = final_df[final_df['first_consequence'].isin(to_filter[key])]
    
    #df_for_analysis = dfs_for_analysis[key]
    #final_df = pd.merge(for_vcf, df_for_analysis, on=['pos_id', 'pos', 'GeneSymbol'], how='inner')[['Name', 'GeneSymbol', 'chrom', 'ClinicalSignificance', 'ReviewStatus', 'pos', 'ref', 'alt', 'hgvs_c', 'aa_change', 'pos_id', 'Consequence','first_consequence','intronic_dist', 'maxSpliceAI', 'splice_impact', 'simple_splice_impact', 'YearLastEvaluated']]

    filtered_df = filtered_df[~((filtered_df['intronic_dist'].isin(['1', '2'])))] #Removes canonical splice variants that may have been misannotated by filtering on distance to coding sequence

    
    if intron_subset:
        filtered_df.loc[(filtered_df['first_consequence'] == 'intron_variant') & (filtered_df['intronic_dist'] >=  50), 'first_consequence'] = 'deep_intron'
        filtered_df.loc[(filtered_df['first_consequence'] == 'intron_variant') & (filtered_df['intronic_dist'] < 50), 'first_consequence'] = 'near_intron'
    
    
    print(f'-----{key}-----')
    
    bar_plot, proportion_df = consequence_bars(filtered_df, key)
    pr_curve = make_pr_curve(filtered_df)
    proportion_map = heatmap(proportion_df)

    if save_plots:
        to_save={'ProportionBars': bar_plot,
                 'PrecisionRecall': pr_curve,
                 'Heatmap': proportion_map
                }

        for item in to_save.keys():
            save_string=f'/net/bbi/vol1/home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/figs/{figure_date_of_record}_{key}_{item}_AllClinVar_IntronsSubsetted.png'

            to_save[item].save(save_string)

-----all_excl_splite_site-----


alt.FacetChart(...)

alt.LayerChart(...)

                     first_consequence ClinicalSignificance  \
0                       intron_variant                  PLP   
0        splice_donor_5th_base_variant                  PLP   
0          splice_donor_region_variant                  PLP   
0  splice_polypyrimidine_tract_variant                  PLP   
0                splice_region_variant                  PLP   
1                   synonymous_variant                  PLP   
1                       intron_variant                  BLB   
1        splice_donor_5th_base_variant                  BLB   
1          splice_donor_region_variant                  BLB   
1  splice_polypyrimidine_tract_variant                  BLB   
1                splice_region_variant                  BLB   
1                   synonymous_variant                  BLB   

      simple_splice_impact  count  proportion  total_count  
0  Splice Impact Predicted     17    0.586207           29  
0  Splice Impact Predicted    130    0.948905          137

alt.LayerChart(...)

...

-----intron_synonymous_only-----


alt.FacetChart(...)

alt.LayerChart(...)

    first_consequence ClinicalSignificance     simple_splice_impact  count  \
0      intron_variant                  PLP  Splice Impact Predicted     17   
1  synonymous_variant                  PLP  Splice Impact Predicted      8   
1      intron_variant                  BLB  Splice Impact Predicted    186   
1  synonymous_variant                  BLB  Splice Impact Predicted    382   

   proportion  total_count  
0    0.586207           29  
1    0.347826           23  
1    0.005582        33320  
1    0.015658        24397  


alt.LayerChart(...)

## Intronic Variant Analysis

In [19]:
intron_df = final_df[(final_df['first_consequence']=='intron_variant') & (~final_df['intronic_dist'].isin(['1','2']))].copy()
intron_df = intron_df[(intron_df['intronic_dist'] < 50) & (intron_df['intronic_dist'] >= 9)].copy()
intron_df['base_change'] = intron_df['hgvs_c'].transform(lambda x: x[-3:])
plp_intron_df = intron_df[intron_df['ClinicalSignificance'] == 'PLP'].copy()

plp_intron_summary_df = pd.concat([plp_intron_df.value_counts(subset='base_change'), plp_intron_df.value_counts(subset='base_change', normalize=True)], axis=1, keys=['count', 'proportion']).reset_index()

In [20]:
plp_base_proportions = alt.Chart(plp_intron_summary_df).mark_bar().encode(
    x = 'base_change:N',
    y='proportion:Q'
).properties(title = 'Proportion of Base Changes in Near Intronic PLPs')

plp_base_proportions.display()

alt.Chart(...)

In [21]:
grouped = intron_df.groupby(['base_change', 'ClinicalSignificance'])

processed_tuples = []

def get_counts(summary, label):
    """Safely extract count + proportion for a given simple_splice_impact label."""
    row = summary[summary['simple_splice_impact'] == label]
    if row.empty:
        return 0, 0.0
    return row['count'].iloc[0], row['proportion'].iloc[0]

processed_tuples = []

for (base_change, clinical_significance), df in grouped:

    summary = pd.concat([
        df.value_counts(subset='simple_splice_impact'),
        df.value_counts(subset='simple_splice_impact', normalize=True)
    ], axis=1, keys=['count', 'proportion']).reset_index()

    if clinical_significance == 'BLB':
        correct_count, correct_proportion = get_counts(summary, '< Threshold')
        incorrect_count, incorrect_proportion = get_counts(summary, 'Splice Impact Predicted')
    else:
        correct_count, correct_proportion = get_counts(summary, 'Splice Impact Predicted')
        incorrect_count, incorrect_proportion = get_counts(summary, '< Threshold')

    processed_tuples.append((clinical_significance, base_change, correct_count, incorrect_count, correct_proportion, incorrect_proportion))

base_change_summary_df = pd.DataFrame(processed_tuples, columns=[
    'clinical_significance', 'base_change', 'correct_count', 'incorrect_count', 'correct_proportion', 'incorrect_proportion'
]).sort_values(['clinical_significance', 'correct_proportion'], ascending=False).reset_index(drop=True)

print(base_change_summary_df.head)

<bound method NDFrame.head of    clinical_significance base_change  correct_count  incorrect_count  \
0                    PLP         A>G              3                0   
1                    PLP         C>A              1                0   
2                    PLP         G>A              1                0   
3                    PLP         T>G              2                0   
4                    PLP         T>A              0                1   
5                    BLB         A>C            130                0   
6                    BLB         G>T            216                1   
7                    BLB         T>C            422                2   
8                    BLB         C>A            177                1   
9                    BLB         G>C            217                2   
10                   BLB         C>T            781                8   
11                   BLB         A>T            126                2   
12                   BLB         T

In [22]:
base_change_heatmap = alt.Chart(base_change_summary_df).mark_rect().encode(
    x=alt.X('clinical_significance:N'),
    y=alt.Y('base_change:N'),
    color=alt.Color('incorrect_proportion:Q'),
    tooltip = 'incorrect_proportion:Q'
)

base_change_heatmap.display()

alt.Chart(...)